# 02 — Search Papers by Keyword via OpenAlex

This notebook demonstrates how to:
1. Search OpenAlex for papers matching a keyword or phrase
2. Filter by open access status, publication year, or type
3. Paginate through results using OpenAlex's cursor-based pagination
4. Collect results into a DataFrame

**API docs:** https://docs.openalex.org/api-entities/works/search-works

In [ ]:
import requests
import pandas as pd
import time

In [ ]:
MAILTO = 'your.email@example.com'  # replace with your e-mail
BASE_URL = 'https://api.openalex.org/works'

## 1. Basic keyword search

In [ ]:
params = {
    'search': 'machine learning africa',
    'per_page': 10,
    'mailto': MAILTO,
}

response = requests.get(BASE_URL, params=params)
data = response.json()

print(f"Total matching works: {data['meta']['count']}")
print(f"Returned this page : {len(data['results'])}")

In [ ]:
# Quick look at titles
for i, work in enumerate(data['results'], 1):
    print(f"{i:2d}. [{work.get('publication_year')}] {work.get('title', 'No title')[:90]}")

## 2. Add filters: open access only, year range

In [ ]:
# Filters are combined with commas; ranges use colons
params_filtered = {
    'search': 'machine learning africa',
    'filter': 'open_access.is_oa:true,publication_year:2020-2023',
    'per_page': 10,
    'mailto': MAILTO,
}

response = requests.get(BASE_URL, params=params_filtered)
data_filtered = response.json()

print(f"Filtered results (OA, 2020-2023): {data_filtered['meta']['count']}")

In [ ]:
for i, work in enumerate(data_filtered['results'], 1):
    oa = work.get('open_access', {}).get('oa_status', 'unknown')
    print(f"{i:2d}. [{work.get('publication_year')}][{oa:8s}] {work.get('title', '')[:80]}")

## 3. Cursor-based pagination

OpenAlex returns a `next_cursor` value in `meta` that you pass as the `cursor` parameter in the next request. Use `cursor=*` to start from the beginning.

In [ ]:
def search_all_works(query: str, filters: str = '', max_results: int = 50, mailto: str = '') -> list:
    """Fetch up to max_results works matching the query, using cursor pagination."""
    results = []
    cursor = '*'

    while len(results) < max_results:
        params = {
            'search': query,
            'per_page': min(25, max_results - len(results)),
            'cursor': cursor,
            'mailto': mailto,
        }
        if filters:
            params['filter'] = filters

        response = requests.get(BASE_URL, params=params)
        data = response.json()
        page_results = data.get('results', [])

        if not page_results:
            break

        results.extend(page_results)
        cursor = data.get('meta', {}).get('next_cursor')

        if not cursor:
            break

        time.sleep(0.2)  # be polite

    print(f'Fetched {len(results)} works')
    return results

In [ ]:
works = search_all_works(
    query='machine learning africa',
    filters='open_access.is_oa:true,publication_year:2020-2023',
    max_results=30,
    mailto=MAILTO,
)

## 4. Collect results into a DataFrame

In [ ]:
def works_to_dataframe(works: list) -> pd.DataFrame:
    rows = []
    for w in works:
        authorships = w.get('authorships', [])
        authors = '; '.join(
            a['author']['display_name']
            for a in authorships[:3]  # first three authors
            if a.get('author')
        )
        rows.append({
            'openalex_id': w.get('id', '').split('/')[-1],
            'doi': w.get('doi', ''),
            'title': w.get('title', ''),
            'year': w.get('publication_year'),
            'cited_by_count': w.get('cited_by_count'),
            'is_oa': w.get('open_access', {}).get('is_oa'),
            'oa_status': w.get('open_access', {}).get('oa_status'),
            'authors_preview': authors,
            'type': w.get('type'),
        })
    return pd.DataFrame(rows)


df = works_to_dataframe(works)
print(df.shape)
df.head(10)

In [ ]:
# Summary statistics
print('OA status breakdown:')
print(df['oa_status'].value_counts())
print()
print('Publication year breakdown:')
print(df['year'].value_counts().sort_index())